# From-Scratch GRPO on OTN — Colab Runner

End-to-end Colab notebook for training the from-scratch GRPO trainer
(`grpo_scratch/`) on a Colab CUDA GPU. Use this when MPS on a Mac is
too slow for autoregressive sampling.

**Recommended Colab runtime:** T4 (free) or L4/A100 (Colab Pro)
- T4 (16 GB): fits Qwen 2.5-1.5B comfortably; Qwen 2.5-3B with bf16 + LoRA + grad checkpointing
- L4 / A100 (24+ GB): fits Qwen 2.5-3B at full speed; Qwen 2.5-7B with grad checkpointing

Set runtime: **Runtime → Change runtime type → GPU**.

## 1. Verify GPU + install deps

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q "torch>=2.2" "transformers>=4.40" "peft>=0.10" "datasets>=2.14" "accelerate>=0.30" pytest
# Colab ships torchao 0.10 which is below the version PEFT's LoRA dispatcher
# requires (>=0.16). We do not use torchao, so uninstalling makes PEFT skip
# that branch cleanly.
%pip uninstall -y torchao

## 2. Clone the repo

In [ ]:
import os, pathlib
REPO_DIR = '/content/OTN-simulator'
if not pathlib.Path(REPO_DIR).exists():
    !git clone https://github.com/barryvv/OTN-simulator.git {REPO_DIR}
%cd {REPO_DIR}
!ls grpo_scratch/

## 3. Upload `train.jsonl`

The training dataset (`outputs/grpo_data/train.jsonl`, ~2.8 MB) is gitignored
in this public repo. Run the cell below and select your local copy.

*Alternative:* mount Google Drive and copy from there. See the optional Drive cell.

In [ ]:
from google.colab import files
uploaded = files.upload()           # pick train.jsonl from local disk
import os, shutil, pathlib
for name in uploaded:
    target = pathlib.Path('outputs/grpo_data/train.jsonl')
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(name, target)
    print(f'[ok] {name} -> {target} ({target.stat().st_size/1e6:.2f} MB)')

In [ ]:
# Optional: load from Drive instead of uploading. Uncomment and adjust the path.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, pathlib
# src = pathlib.Path('/content/drive/MyDrive/path/to/train.jsonl')
# dst = pathlib.Path('outputs/grpo_data/train.jsonl')
# dst.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy(src, dst)
# print('[ok] copied:', dst, dst.stat().st_size)

## 4. Run the unit tests (proof the math is right)

Six tests covering the group-relative advantage standardization,
clipped surrogate loss, KL toggle, and log-prob shift+mask.

In [ ]:
!python -m pytest tests/test_grpo_scratch.py -v

## 5. Train

Defaults below target a T4 (16 GB) with Qwen 2.5-3B + LoRA + bf16 +
gradient checkpointing. Tune `--group-size`, `--max-new-tokens`, and
`--max-steps` based on your runtime.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train_grpo_scratch.py \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --train-file outputs/grpo_data/train.jsonl \
  --output-dir /content/adapters/qwen-3b-grpo-scratch \
  --device cuda \
  --dtype bfloat16 \
  --group-size 4 \
  --max-new-tokens 1024 \
  --max-steps 50 \
  --lr 5e-5 \
  --temperature 1.3 \
  --max-grad-norm 10.0 \
  --reward-fn continuous \
  --gradient-checkpointing \
  --empty-cache-between-phases

### Optional: bigger model (L4 / A100 only)

If you have a Colab Pro L4 (24 GB) or A100 (40 GB), swap to Qwen 2.5-7B
for the production target. Keep `--gradient-checkpointing` on the L4.

In [ ]:
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train_grpo_scratch.py \
#   --base-model Qwen/Qwen2.5-7B-Instruct \
#   --train-file outputs/grpo_data/train.jsonl \
#   --output-dir /content/adapters/qwen-7b-grpo-scratch \
#   --device cuda --dtype bfloat16 \
#   --group-size 4 --max-new-tokens 1024 --max-steps 50 --lr 5e-6 --temperature 1.0 \
#   --gradient-checkpointing --empty-cache-between-phases

## 6. Download or persist the adapter

Two paths: zip and download to your laptop, or copy to Drive.

In [ ]:
import shutil, pathlib
adapter_dir = pathlib.Path('/content/adapters/qwen-3b-grpo-scratch/final')
if adapter_dir.exists():
    zip_path = shutil.make_archive('/content/grpo-scratch-final', 'zip', adapter_dir)
    from google.colab import files
    files.download(zip_path)
else:
    print(f'[warn] {adapter_dir} not found; training may still be running.')

In [ ]:
# Optional: copy adapter to Drive.
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, pathlib
# src = pathlib.Path('/content/adapters/qwen-3b-grpo-scratch/final')
# dst = pathlib.Path('/content/drive/MyDrive/grpo-scratch-final')
# shutil.copytree(src, dst, dirs_exist_ok=True)
# print('[ok] saved to', dst)